In [17]:
import litellm
import pandas as pd 
from src.prompts import BASE_PROMPT
from sklearn.metrics import classification_report
import numpy as np
from src.metrics import recall_at_k, page_score_np
from src.utils import load_pdf_with_plumber
from src.retrievers import get_embeddings, create_nmslib_index, ann_search, bm25_search, create_bm25_retriever
import os
from tqdm import tqdm
import dotenv
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL2= "intfloat/multilingual-e5-base"

dirs = ["data/domain_1/dev", "data/domain_2/dev"]

pages = []
doc_id = 0
filename_to_doc_id = {}  # Mapping from filename to doc_id

for dir in dirs:
    for filename in os.listdir(dir):
        if filename.endswith(".pdf"):
            result = load_pdf_with_plumber(pdf_path=os.path.join(dir, filename), doc_id=doc_id)
            domain = "domain_2"
            if "domain_1" in dir:
                domain = "domain_1"
            result = [(doc_id, page_num, text, domain) for (doc_id, page_num, text) in result]
            pages.extend(result)
            filename_to_doc_id[filename] = doc_id
            doc_id += 1

print(f"Loaded {len(pages)} pages from {doc_id} documents.")

df_corpus = pd.DataFrame(pages, columns=['Doc_ID', 'Page_Num', 'Text_Content', 'Domain'])
df = pd.read_csv("data/dev_questions.csv")
df['options'] = df[['A', 'B', 'C', 'D', 'E', 'F']].apply(
    lambda row: ' '.join([f"{chr(65 + i)}. {option}" for i, option in enumerate(row)]), axis=1
)


Loaded 1121 pages from 41 documents.


In [6]:
emb_model = SentenceTransformer(EMBEDDING_MODEL2, trust_remote_code=True)

In [43]:
from src.chunking import (
    fixed_size_chunking,
)
from functools import partial


def create_chunked_df(df_corpus, chunking_strategy) -> pd.DataFrame:
    """Create chunked dataframe from corpus using given strategy."""
    rows = [
        {'Doc_ID': row.Doc_ID, 'Page_Num': row.Page_Num, 'Text_Content': chunk, "Domain": row.Domain}
        for row in df_corpus.itertuples()
        for chunk in chunking_strategy(row.Text_Content)
        if chunk.strip()  # Skip empty chunks
    ]
    return pd.DataFrame(rows)

df_fixed_600 = create_chunked_df(
    df_corpus, partial(fixed_size_chunking, chunk_size=600, overlap=100)
)

pages_embeddings2 = get_embeddings(df_fixed_600['Text_Content'].tolist(), model=emb_model)
nmslib_index = create_nmslib_index(pages_embeddings2)
retriever = create_bm25_retriever(df_fixed_600['Text_Content'].tolist())

def reciprocal_rank_fusion(embedding_ranks, bm25_ranks, alpha=0.5):
    combined_scores = {}
    for rank, idx in enumerate(embedding_ranks):
        combined_scores[idx] = combined_scores.get(idx, 0) + alpha / (rank + 1)
    for rank, idx in enumerate(bm25_ranks):
        combined_scores[idx] = combined_scores.get(idx, 0) + (1 - alpha) / (rank + 1)
    # Sort by combined score
    sorted_indices = sorted(combined_scores.keys(), key=lambda x: combined_scores[x], reverse=True)
    return sorted_indices

def search_with_domain_filter(query, top_k=5, domain=None):
    indicies, _ = ann_search(query, nmslib_index, emb_model, top_k=top_k*10)
    indicies_bm25 = bm25_search(query, retriever, top_k=top_k*10)

    indicies = reciprocal_rank_fusion(indicies, indicies_bm25, alpha=0.5)
    results = df_fixed_600.iloc[indicies]
    if domain:
        results = results[results['Domain'] == domain]

    return results.head(top_k)["Text_Content"].tolist()


Batches: 100%|██████████| 262/262 [00:10<00:00, 25.15it/s]


In [87]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="Qwen/Qwen3-VL-8B-Instruct-FP8",
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": False,
        "structured_output": True,
        "family": "qwen"
    },
)


async def find_chunks(query: str, domain: str) -> str:
    """Find relevant chunks for a given query."""
    chunks = search_with_domain_filter(query, top_k=5, domain=domain)
    result = ""
    for i, chunk in enumerate(chunks):
        result += f"Chunk {i+1}:\n{chunk}\n\n"
    return result

PROMPT = """
You are a question answering agent. You have to answer on user question with one letter (A, B, C, D, E, F). 
You may use the tool to find relevant document chunks based on the user's query and domain.
If you use the tool, you can provide the domain as 'domain_1' or 'domain_2', where domain_2 is medical domain, and domain_1 is everything else.
Make sure to use the tool.
"""

PROMPT_USER = """
Question: {question}
Options: {options}
Answer (Only option letter):


"""


agent = AssistantAgent(
    name="rag_agent",
    model_client=model_client,
    tools=[find_chunks],
    system_message=PROMPT,
    reflect_on_tool_use=True,
)

In [ ]:
answers_rag = []

options_letters = [chr(65 + j) for j in range(6)]  # A-F

for _, row in tqdm(df.iterrows()):
    question = row["Question"]
    options = row["options"]
    correct_answer = row["Correct_Answer"]

    # Retrieve context passages
    user_prompt = PROMPT_USER.format(question=question, options=options)
    response = await agent.run(task=user_prompt)
    answer = response.messages[-1].content
    if answer not in options_letters:
        answer = "A"
    answers_rag.append(answer)
    await agent.on_reset(cancellation_token=None)

461it [09:31,  1.24s/it]


In [ ]:
print(classification_report(answers_rag, df["Correct_Answer"]))

              precision    recall  f1-score   support

           A       0.66      0.84      0.74        62
           B       0.82      0.68      0.74        81
           C       0.77      0.69      0.73        80
           D       0.77      0.79      0.78        75
           E       0.68      0.80      0.74        66
           F       0.81      0.74      0.77        97

    accuracy                           0.75       461
   macro avg       0.75      0.76      0.75       461
weighted avg       0.76      0.75      0.75       461



In [90]:
from ddgs import DDGS
PROMPT_SEARCH = """
You are a question answering agent. You have to answer on user question.
You have to use the tools to find relevant document chunks based on the user's query and domain.
If you use the find_chunks tool, you can provide the domain as 'domain_1' or 'domain_2', where domain_2 is medical domain, and domain_1 is everything else.
Use both tools to find additional information to answer the question.
Final answer should be only one letter of the provided options (A, B, C, D, E, F).
"""

PROMPT_USER_SEARCH = """
Question: {question}
Options: {options}
Answer (Only option letter):
"""

async def search_web_tool(query: str) -> str:
    results = DDGS().text(query, max_results=5)
    return results

agent_with_search = AssistantAgent(
    name="rag_agent",
    model_client=model_client,
    tools=[find_chunks, search_web_tool],
    system_message=PROMPT_SEARCH,
    reflect_on_tool_use=True,
    max_tool_iterations=2,
)

answers_rag_with_search = []

options_letters = [chr(65 + j) for j in range(6)]  # A-F

for _, row in tqdm(df.iterrows()):
    question = row["Question"]
    options = row["options"]
    correct_answer = row["Correct_Answer"]

    # Retrieve context passages
    user_prompt = PROMPT_USER_SEARCH.format(question=question, options=options)
    response = await agent_with_search.run(task=user_prompt)
    answer = response.messages[-1].content
    if answer not in options_letters:
        answer = "A"
    answers_rag_with_search.append(answer)
    await agent_with_search.on_reset(cancellation_token=None)

print(classification_report(answers_rag_with_search, df["Correct_Answer"]))

461it [08:27,  1.10s/it]

              precision    recall  f1-score   support

           A       0.70      0.73      0.71        75
           B       0.75      0.66      0.70        76
           C       0.77      0.70      0.73        79
           D       0.69      0.75      0.72        71
           E       0.68      0.82      0.74        65
           F       0.80      0.75      0.77        95

    accuracy                           0.73       461
   macro avg       0.73      0.73      0.73       461
weighted avg       0.74      0.73      0.73       461



# OpenAI model

I tried swapping model to gpt-5-mini, but this did not show significant improvement either, this is basically the same as default RAG. 
I would assume the reason is because this has little to no changes compared to rag, but introduces additional complexity -> possibly increases error. In my scenario the only thing that differs is filtering by domain, but I am really not sure that this is that important, from my previous tests I noticed that when searching for something in one domain fused indicies rarely give something from another domain. 

Adding duckduckgo search is cool and so on, but duckduckgo is not that good search tool from my previous experience, maybe it could give some improvement but would require additional tweaking, maybe searching in english, maybe making multiple requests, but search tool alone does not seem to improve anything.


In [ ]:
model_client = OpenAIChatCompletionClient(
    model="gpt-5-mini",
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": False,
        "structured_output": True,
        "family": "openai"
    },
)

agent = AssistantAgent(
    name="rag_agent",
    model_client=model_client,
    tools=[find_chunks],
    system_message=PROMPT,
    reflect_on_tool_use=True,
)
answers_rag = []

options_letters = [chr(65 + j) for j in range(6)]  # A-F

for _, row in tqdm(df.iterrows()):
    question = row["Question"]
    options = row["options"]
    correct_answer = row["Correct_Answer"]

    # Retrieve context passages
    user_prompt = PROMPT_USER.format(question=question, options=options)
    response = await agent.run(task=user_prompt)
    answer = response.messages[-1].content
    if answer not in options_letters:
        answer = "A"
    answers_rag.append(answer)
    await agent.on_reset(cancellation_token=None)

In [96]:
print(classification_report(answers_rag, df["Correct_Answer"][:len(answers_rag)]))

              precision    recall  f1-score   support

           A       0.97      0.61      0.75       127
           B       0.80      0.88      0.84        60
           C       0.82      0.92      0.87        63
           D       0.81      0.91      0.86        68
           E       0.81      0.95      0.88        66
           F       0.84      0.99      0.91        76

    accuracy                           0.84       460
   macro avg       0.84      0.88      0.85       460
weighted avg       0.86      0.84      0.84       460

